In [0]:
from pyspark.sql.functions import expr,col,sum

bookings_df = spark.table("dev.spark_db.bookings").alias("b")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")
members_df = spark.table("dev.spark_db.members").alias("m")


df = bookings_df.join(facilities_df , expr("b.facility_id == f.facility_id") , 'inner')\
                .join(members_df , expr("m.member_id == b.member_id") , 'left')\
                .where("month(start_time) == 7 and year(start_time) == 2022")\
                .withColumns(
                    {
                        "booked_by" : expr("case when m.member_id == 0 then 'Guest' else 'Member' end"),
                        "booking_date" : expr("to_date(start_time)"),
                        "booking_amount" : expr("case when m.member_id == 0 then slots * guest_cost else slots * member_cost end")
                    }
                )\
                .groupBy(col("booked_by") , col("booking_date"))\
                    .agg(sum(col("booking_amount")).alias("revenue"))\
                .orderBy(col("booking_date").asc())
df.display()